Cell 1: Import Library & Persiapan Data Query

In [1]:
# CELL 1: Import Library dan Load Query Excel
import requests
import pandas as pd
import time
import os

# Karena query.xlsx satu folder dengan notebook ini, kita panggil langsung
df_query = pd.read_excel('query.xlsx')

# Mapping nama kategori Excel ke format kolom ADUIN
kategori_mapping = {
    "Infrastruktur & Jalan": "kategori_infrastruktur",
    "Lingkungan & Sampah": "kategori_lingkungan",
    "Air & Sanitasi": "kategori_air_sanitasi",
    "Bencana & Cuaca": "kategori_bencana",
    "Transportasi": "kategori_transportasi",
    "Pelayanan Publik": "kategori_pelayanan_publik",
    "Ekonomi & UMKM": "kategori_ekonomi",
    "Keamanan & Sosial": "kategori_keamanan",
    "Pendidikan": "kategori_pendidikan",
    "Kesehatan": "kategori_kesehatan"
}

kategori_queries = {}
for index, row in df_query.iterrows():
    nama_excel = row['Kategori'].strip()
    kata_kunci = row['Rekomendasi Kata Kunci (Keywords)']
    
    nama_kolom_aduin = kategori_mapping.get(nama_excel)
    if nama_kolom_aduin:
        # Menambahkan filter bahasa dan mengecualikan retweet
        query_api = f"({kata_kunci}) lang:id -is:retweet"
        kategori_queries[nama_kolom_aduin] = query_api

print("✅ Query pencarian berhasil dirakit!")

✅ Query pencarian berhasil dirakit!


Cell 2: Eksekusi Scraping X API

In [2]:
# CELL 2: Eksekusi Scraping API X
import os
from dotenv import load_dotenv

# Memuat variabel dari file .env
# Gunakan override=True agar selalu membaca nilai terbaru jika .env diubah
load_dotenv(override=True)

# Mengambil token dari sistem dengan aman
BEARER_TOKEN = os.getenv("BEARER_TOKEN_X")

if not BEARER_TOKEN:
    raise ValueError("❌ BEARER_TOKEN_X tidak ditemukan! Cek kembali file .env kamu.")
else:
    print("✅ Token berhasil diamankan dan dimuat!")

def scrape_x_api(kategori_queries, max_results_per_query=100):
    all_data = []
    # Memasukkan Bearer Token ke dalam header otentikasi
    headers = {"Authorization": f"Bearer {BEARER_TOKEN}"}
    search_url = "https://api.twitter.com/2/tweets/search/recent"

    for nama_kategori, query in kategori_queries.items():
        print(f"🔄 Mengambil data untuk: {nama_kategori}...")
        
        params = {
            'query': query,
            'max_results': max_results_per_query, 
            'tweet.fields': 'created_at,author_id' 
        }

        try:
            response = requests.get(search_url, headers=headers, params=params)
            
            if response.status_code == 200:
                json_response = response.json() 
                
                if 'data' in json_response:
                    for tweet in json_response['data']:
                        row = {
                            'waktu': tweet.get('created_at'),
                            'teks': tweet.get('text'),
                            nama_kategori: 1
                        }
                        all_data.append(row)
                    print(f"  ✅ Berhasil mendapat {len(json_response['data'])} keluhan.")
                else:
                    print(f"  ⚠️ Tidak ada data keluhan terbaru.")
            else:
                print(f"  ❌ Request Error: {response.status_code} - {response.text}")
                
        except Exception as e:
            print(f"  ❌ Terjadi kesalahan koneksi: {e}")
        
        # Jeda aman agar tidak terkena limit API
        time.sleep(3) 
        
    return all_data

# Jalankan fungsi scraping
hasil_scraping = scrape_x_api(kategori_queries, max_results_per_query=100)

✅ Token berhasil diamankan dan dimuat!
🔄 Mengambil data untuk: kategori_infrastruktur...
  ❌ Request Error: 402 - {"account_id":2051482604390526981,"title":"CreditsDepleted","detail":"Your enrolled account [2051482604390526981] does not have any credits to fulfill this request.","type":"https://api.twitter.com/2/problems/credits"}
🔄 Mengambil data untuk: kategori_lingkungan...
  ❌ Request Error: 402 - {"account_id":2051482604390526981,"title":"CreditsDepleted","detail":"Your enrolled account [2051482604390526981] does not have any credits to fulfill this request.","type":"https://api.twitter.com/2/problems/credits"}
🔄 Mengambil data untuk: kategori_air_sanitasi...
  ❌ Request Error: 402 - {"account_id":2051482604390526981,"title":"CreditsDepleted","detail":"Your enrolled account [2051482604390526981] does not have any credits to fulfill this request.","type":"https://api.twitter.com/2/problems/credits"}
🔄 Mengambil data untuk: kategori_bencana...
  ❌ Request Error: 402 - {"account_id":